# LeetCode 68 — Text Justification (Căn đều văn bản) bằng Python

Notebook này giảng nhập môn bài **Hard**:

```python
class Solution:
    def fullJustify(self, words: List[str], maxWidth: int) -> List[str]:

```

Đây là một bài **mô phỏng (simulation)**: không có công thức thần kỳ, cũng không có "brute force" kiểu thử mọi khả năng. Cái khó nằm ở chỗ **làm đúng từng chi tiết nhỏ**: chia dòng, đếm khoảng trắng, dồn khoảng thừa, xử lý dòng cuối và dòng chỉ có 1 từ.

Mục tiêu của notebook:

- Hiểu chính xác đề yêu cầu gì (căn đều hai bên giống Microsoft Word).
- Học các công cụ chuỗi của Python từ căn bản: `*` lặp chuỗi, `str.join`, `str.ljust`, `divmod`, toán tử `%` và mẹo `x or y`.
- Biết **chia nhỏ vấn đề** thành 3 bước: chia dòng → phân phối khoảng trắng → xử lý trường hợp đặc biệt.
- Cài đặt 3 cách: bản tách hàm rõ ràng (cách chính), bản gọn **round-robin**, bản gọn dùng chỉ số + `ljust`/`join`.
- Tự kiểm tra bằng một hàm "trọng tài" viết theo đúng định nghĩa của đề, test ngẫu nhiên.

## 1. Đề bài

Cho một mảng chuỗi `words` và một số nguyên `maxWidth`. Hãy định dạng văn bản sao cho **mỗi dòng có đúng `maxWidth` ký tự** và được **căn đều hai bên** (fully justified — căn thẳng cả lề trái lẫn lề phải).

Quy tắc của đề:

1. Xếp từ theo kiểu **greedy** (tham lam): mỗi dòng nhét **càng nhiều từ càng tốt**. Thêm dấu cách `' '` khi cần để mỗi dòng có đúng `maxWidth` ký tự.
2. Khoảng trắng thừa giữa các từ phải chia **đều nhất có thể**. Nếu số khoảng trắng không chia hết cho số khe giữa các từ, thì **các khe bên trái nhận nhiều hơn** các khe bên phải.
3. **Dòng cuối cùng** được **căn trái** (left-justified): giữa các từ chỉ có đúng 1 dấu cách, phần còn lại đệm dấu cách ở cuối dòng.

Ghi chú của đề:

- Một "từ" là dãy ký tự **không chứa dấu cách**.
- Độ dài mỗi từ luôn `> 0` và **không vượt quá** `maxWidth`.
- Mảng `words` có **ít nhất một từ**.

### Ví dụ của đề

**Ví dụ 1**

```text
Input:  words = ["This", "is", "an", "example", "of", "text", "justification."], maxWidth = 16
Output:
[
   "This    is    an",
   "example  of text",
   "justification.  "
]
```

**Ví dụ 2**

```text
Input:  words = ["What","must","be","acknowledgment","shall","be"], maxWidth = 16
Output:
[
  "What   must   be",
  "acknowledgment  ",
  "shall be        "
]
```

Giải thích của đề: dòng cuối là `"shall be        "` chứ **không phải** `"shall         be"`, vì dòng cuối phải căn trái. Dòng thứ hai cũng được căn trái vì nó **chỉ có một từ**.

**Ví dụ 3**

```text
Input:  words = ["Science","is","what","we","understand","well","enough","to","explain","to","a","computer.","Art","is","everything","else","we","do"], maxWidth = 20
Output:
[
  "Science  is  what we",
  "understand      well",
  "enough to explain to",
  "a  computer.  Art is",
  "everything  else  we",
  "do                  "
]
```

### Constraints

- `1 <= words.length <= 300`
- `1 <= words[i].length <= 20`
- `words[i]` chỉ gồm chữ cái tiếng Anh và ký hiệu.
- `1 <= maxWidth <= 100`
- `words[i].length <= maxWidth`

### Mẹo nhìn thấy khoảng trắng: bọc dòng bằng `|`

Dấu cách là ký tự "vô hình", nên khi in ra rất khó đếm. Trong cả notebook, ta sẽ in mỗi dòng kẹp giữa hai dấu `|`:

```text
|This    is    an|
```

Nhờ vậy ta thấy rõ: dòng bắt đầu ở đâu, kết thúc ở đâu, và có bao nhiêu dấu cách ở cuối dòng.

Hàm `show` dưới đây in từng dòng kèm độ dài của nó.

In [1]:
from typing import List


def show(lines: List[str]) -> None:
    for line in lines:
        print(f"|{line}|  len = {len(line)}")


expected_1 = [
    "This    is    an",
    "example  of text",
    "justification.  ",
]
show(expected_1)

|This    is    an|  len = 16
|example  of text|  len = 16
|justification.  |  len = 16


Quan sát kết quả mong đợi của Ví dụ 1:

| Dòng | Các từ | Tổng chữ cái | Số khe | Khoảng trắng mỗi khe | Ghi chú |
|---|---|---:|---:|---|---|
| 1 | `This is an` | 4+2+2 = 8 | 2 | 4, 4 | chia hết, đều nhau |
| 2 | `example of text` | 7+2+4 = 13 | 2 | 2, 1 | dư 1 → khe **trái** nhận thêm |
| 3 | `justification.` | 14 | 0 | — | dòng cuối: căn trái, đệm 2 dấu cách ở cuối |

Bảng này chính là "bản thiết kế" của thuật toán: ta cần biết **dòng gồm những từ nào**, **tổng chữ cái**, **số khe** và **cách chia khoảng trắng**.

## 2. Kiến thức Python nền tảng về chuỗi

Trước khi giải, ta học 5 công cụ sẽ dùng liên tục. Mỗi công cụ có một ô code nhỏ để bạn tự chạy và quan sát.

### 2.1. `len` và phép `*` lặp chuỗi

- `len(s)` trả về số ký tự của chuỗi `s`.
- `s * k` (với `k` là số nguyên) tạo chuỗi mới gồm `s` lặp lại `k` lần.
- `' ' * k` là cách nhanh nhất để tạo **đúng `k` dấu cách**. Nếu `k = 0` thì được chuỗi rỗng `''`.

Ngoài ra `sum(len(w) for w in line)` cộng độ dài của mọi từ trong `line` (đọc là: "tổng của `len(w)` với mỗi `w` trong `line`").

In [2]:
print(len("example"))          # 7
print("|" + "ab" * 3 + "|")    # |ababab|
print("|" + " " * 4 + "|")     # |    |  (4 dấu cách)
print("|" + " " * 0 + "|")     # ||      (0 dấu cách = chuỗi rỗng)

line = ["This", "is", "an"]
print(sum(len(w) for w in line))   # 4 + 2 + 2 = 8

7
|ababab|
|    |
||
8


### 2.2. `str.join` — nối các từ bằng một chuỗi ngăn cách

Cú pháp:

```python
separator.join(list_of_strings)
```

Đọc là: "lấy `separator` dán **vào giữa** các phần tử của list".

- `' '.join(['to', 'be'])` → `'to be'` (1 dấu cách ở giữa).
- `''.join(['a', 'b', 'c'])` → `'abc'` (dán sát nhau).
- Nếu list chỉ có **1 phần tử**, `join` trả lại đúng phần tử đó — không thêm gì.

Điểm quan trọng cho bài này: list có `k` từ thì `join` chèn đúng **`k - 1`** chuỗi ngăn cách. Số `k - 1` chính là **số khe** giữa các từ.

In [3]:
words_demo = ["This", "is", "an"]
print("|" + " ".join(words_demo) + "|")     # |This is an|
print("|" + "".join(words_demo) + "|")      # |Thisisan|
print("|" + "---".join(words_demo) + "|")   # |This---is---an|
print("|" + " ".join(["alone"]) + "|")      # |alone|  -> 1 từ: không có khe
print("số khe =", len(words_demo) - 1)

|This is an|
|Thisisan|
|This---is---an|
|alone|
số khe = 2


### 2.3. `divmod` — chia lấy thương và số dư cùng lúc

- `a // b` là **chia lấy phần nguyên** (thương).
- `a % b` là **chia lấy dư**.
- `divmod(a, b)` trả về cặp `(a // b, a % b)` trong một lần gọi.

Ví dụ: có `5` dấu cách cần chia cho `3` khe:

```text
5 = 3 * 1 + 2   →   divmod(5, 3) = (1, 2)
```

Nghĩa là: mỗi khe nhận **ít nhất 1** dấu cách, còn **dư 2** dấu cách. Theo đề, 2 dấu cách dư này đưa cho **2 khe bên trái**, mỗi khe thêm 1.

Chú ý: `divmod(a, 0)` gây lỗi `ZeroDivisionError` — ta sẽ gặp lại chuyện này ở dòng chỉ có 1 từ.

In [4]:
print(5 // 3, 5 % 3)    # 1 2
print(divmod(5, 3))     # (1, 2)
print(divmod(8, 2))     # (4, 0)  -> chia hết
print(divmod(3, 2))     # (1, 1)

q, r = divmod(5, 3)     # "mở gói" cặp kết quả vào 2 biến
print("q =", q, "| r =", r)

1 2
(1, 2)
(4, 0)
(1, 1)
q = 1 | r = 2


### 2.4. `str.ljust` — căn trái và đệm dấu cách bên phải

`s.ljust(width)` trả về chuỗi `s` được **đệm thêm dấu cách ở bên phải** cho tới khi đủ `width` ký tự.

- Nếu `len(s) >= width` thì trả lại `s` nguyên vẹn (không cắt bớt).
- Tương đương với `s + ' ' * (width - len(s))` khi `width >= len(s)`.

Đây đúng là thao tác "căn trái" mà đề yêu cầu cho dòng cuối.

In [5]:
s = "shall be"
print("|" + s.ljust(16) + "|")                  # |shall be        |
print("|" + s + " " * (16 - len(s)) + "|")      # cùng kết quả
print("|" + "abcdef".ljust(3) + "|")            # |abcdef| -> không cắt
print(len(s.ljust(16)))                         # 16

|shall be        |
|shall be        |
|abcdef|
16


### 2.5. Toán tử `%` để "đi vòng" và mẹo `x or y`

**Đi vòng (round-robin)**: khi `i` tăng dần `0, 1, 2, 3, 4, ...` thì `i % 3` cho `0, 1, 2, 0, 1, ...` — tức là lần lượt chỉ vào 3 vị trí, hết thì quay lại đầu. Ta sẽ dùng nó để "rải" từng dấu cách vào từng khe.

**Mẹo `x or y`**: trong Python, `x or y` trả về `x` nếu `x` là "truthy" (khác 0, khác rỗng), ngược lại trả về `y`.

- `2 or 1` → `2`
- `0 or 1` → `1`

Vì vậy `len(line) - 1 or 1` nghĩa là: "số khe, nhưng nếu số khe bằng 0 thì dùng 1" — một cách gọn để **tránh chia cho 0** (`i % 0` cũng lỗi `ZeroDivisionError`).

In [6]:
print([i % 3 for i in range(7)])   # [0, 1, 2, 0, 1, 2, 0]

print(2 or 1)    # 2
print(0 or 1)    # 1

for line in (["a", "b", "c"], ["alone"]):
    k = len(line) - 1 or 1
    print(line, "-> số dùng để % là", k)

[0, 1, 2, 0, 1, 2, 0]
2
1
['a', 'b', 'c'] -> số dùng để % là 2
['alone'] -> số dùng để % là 1


## 3. Chia nhỏ bài toán thành 3 bước

Bài này nhìn thì rối, nhưng nếu tách ra thì mỗi phần đều đơn giản:

```text
words ──► [Bước 1] chia dòng (greedy)
             │      [["This","is","an"], ["example","of","text"], ["justification."]]
             ▼
          [Bước 2] với mỗi dòng bình thường: phân phối khoảng trắng bằng divmod
             │      "This    is    an", "example  of text"
             ▼
          [Bước 3] trường hợp đặc biệt: dòng cuối, dòng chỉ 1 từ → căn trái
                    "justification.  "
```

Vì sao nên tách? Mỗi bước có **một câu hỏi duy nhất**:

1. Bước 1: dòng này gồm **những từ nào**? (chưa quan tâm khoảng trắng đặt ở đâu)
2. Bước 2: biết các từ rồi, mỗi khe được **bao nhiêu dấu cách**?
3. Bước 3: khi nào **không** căn đều mà căn trái?

Ta sẽ viết mỗi bước thành một hàm riêng, test riêng, rồi ghép lại.

## 4. Bước 1 — Chia dòng theo greedy

**Greedy** (tham lam) ở đây nghĩa là: duyệt từ trái sang phải, **còn nhét được thì cứ nhét** vào dòng hiện tại; nhét không được nữa thì đóng dòng lại và mở dòng mới. Đề bắt buộc cách xếp này, nên ta không cần chứng minh nó "tối ưu" — chỉ cần làm đúng.

### Điều kiện "còn nhét được"

Giả sử dòng hiện tại `cur` đang có `k = len(cur)` từ, tổng số chữ cái là `letters`. Ta muốn thêm từ `w`.

Sau khi thêm, dòng có `k + 1` từ, tức là `k` khe. Mỗi khe cần **tối thiểu 1 dấu cách**. Vậy độ dài **nhỏ nhất** của dòng mới là:

```text
letters + len(w) + k        (chữ cái cũ + chữ cái mới + k dấu cách tối thiểu)
```

- Nếu `letters + len(w) + k <= maxWidth` → nhét được.
- Ngược lại → đóng dòng `cur`, từ `w` mở dòng mới.

Lỗi kinh điển là quên `+ k` (quên các dấu cách tối thiểu) — xem phần lỗi thường gặp.

### Tính tay một lần

Ví dụ 1, `maxWidth = 16`. Dòng đang là `["This", "is", "an"]`:

- `letters = 4 + 2 + 2 = 8`, `k = 3`.
- Muốn thêm `"example"` (7 chữ): độ dài tối thiểu `8 + 7 + 3 = 18 > 16` → **không nhét được**.

Kiểm tra lại bằng code:

In [7]:
cur = ["This", "is", "an"]
letters = sum(len(x) for x in cur)
w = "example"
need = letters + len(w) + len(cur)
print("letters =", letters, "| k =", len(cur), "| need =", need)
print("nhét được?", need <= 16)
print("thử ghép tối thiểu:", "|" + " ".join(cur + [w]) + "|", "len =", len(" ".join(cur + [w])))

letters = 8 | k = 3 | need = 18
nhét được? False
thử ghép tối thiểu: |This is an example| len = 18


### Code Bước 1: `split_into_lines`

Ý tưởng:

- `cur` là list các từ của dòng đang xây, `letters` là tổng chữ cái của `cur`.
- Với mỗi từ `w`: nếu thêm vào bị tràn → đẩy `cur` vào `lines`, reset `cur`.
- Sau đó luôn thêm `w` vào `cur` (vào dòng cũ, hoặc dòng mới vừa mở).
- **Đừng quên**: hết vòng lặp, dòng cuối vẫn nằm trong `cur` — phải đẩy nó vào `lines`.

Vì mỗi từ dài `<= maxWidth` (đề đảm bảo), một dòng mới luôn chứa được ít nhất 1 từ, nên `cur` không bao giờ rỗng khi bị đẩy vào `lines`.

In [8]:
def split_into_lines(words: List[str], maxWidth: int) -> List[List[str]]:
    lines = []
    cur = []        # các từ của dòng đang xây
    letters = 0     # tổng số chữ cái trong cur

    for w in words:
        # len(cur) = số khe sẽ có nếu thêm w (mỗi khe tối thiểu 1 dấu cách)
        if letters + len(w) + len(cur) > maxWidth:
            lines.append(cur)
            cur, letters = [], 0
        cur.append(w)
        letters += len(w)

    lines.append(cur)   # đừng quên dòng cuối
    return lines


words_1 = ["This", "is", "an", "example", "of", "text", "justification."]
for line in split_into_lines(words_1, 16):
    print(line)

['This', 'is', 'an']
['example', 'of', 'text']
['justification.']


### Trace Bước 1 trên Ví dụ 1 (`maxWidth = 16`)

`need = letters + len(w) + len(cur)` là độ dài tối thiểu nếu thêm `w`.

| w | cur trước | letters | len(cur) | need | Quyết định |
|---|---|---:|---:|---:|---|
| `This` | `[]` | 0 | 0 | 4 | thêm |
| `is` | `[This]` | 4 | 1 | 7 | thêm |
| `an` | `[This, is]` | 6 | 2 | 10 | thêm |
| `example` | `[This, is, an]` | 8 | 3 | 18 | **18 > 16 → đóng dòng**, mở dòng mới |
| `of` | `[example]` | 7 | 1 | 10 | thêm |
| `text` | `[example, of]` | 9 | 2 | 15 | thêm |
| `justification.` | `[example, of, text]` | 13 | 3 | 30 | **đóng dòng**, mở dòng mới |
| (hết) | `[justification.]` | 14 | — | — | đẩy dòng cuối |

Hàm `trace_split_lines` in đúng các cột này:

In [9]:
def trace_split_lines(words: List[str], maxWidth: int) -> List[List[str]]:
    lines, cur, letters = [], [], 0
    for w in words:
        need = letters + len(w) + len(cur)
        if need > maxWidth:
            print(f"w={w!r:17} cur={cur} letters={letters} need={need} > {maxWidth} -> ĐÓNG DÒNG")
            lines.append(cur)
            cur, letters = [], 0
        else:
            print(f"w={w!r:17} cur={cur} letters={letters} need={need} -> thêm")
        cur.append(w)
        letters += len(w)
    print("Hết từ, đẩy dòng cuối:", cur)
    lines.append(cur)
    return lines


_ = trace_split_lines(words_1, 16)

w='This'            cur=[] letters=0 need=4 -> thêm
w='is'              cur=['This'] letters=4 need=7 -> thêm
w='an'              cur=['This', 'is'] letters=6 need=10 -> thêm
w='example'         cur=['This', 'is', 'an'] letters=8 need=18 > 16 -> ĐÓNG DÒNG
w='of'              cur=['example'] letters=7 need=10 -> thêm
w='text'            cur=['example', 'of'] letters=9 need=15 -> thêm
w='justification.'  cur=['example', 'of', 'text'] letters=13 need=30 > 16 -> ĐÓNG DÒNG
Hết từ, đẩy dòng cuối: ['justification.']


## 5. Bước 2 — Phân phối khoảng trắng cho một dòng bình thường

Giả sử dòng có `k >= 2` từ và **không phải dòng cuối**. Ta tính:

```text
letters = tổng chữ cái
spaces  = maxWidth - letters      (tổng số dấu cách phải chèn)
gaps    = k - 1                   (số khe giữa các từ)
q, r    = divmod(spaces, gaps)
```

- Mỗi khe nhận **ít nhất `q`** dấu cách.
- Còn dư `r` dấu cách (`0 <= r < gaps`): đưa cho **`r` khe đầu tiên** (bên trái), mỗi khe thêm 1.
- Nên: khe thứ `i` (đếm từ 0) có `q + 1` dấu cách nếu `i < r`, ngược lại có `q`.

Kiểm tra tổng: `r * (q + 1) + (gaps - r) * q = gaps * q + r = spaces`. Đúng bằng số dấu cách cần chèn, nên dòng dài đúng `maxWidth`.

### Tính tay: dòng `["a", "computer.", "Art", "is"]`, `maxWidth = 20` (Ví dụ 3)

- `letters = 1 + 9 + 3 + 2 = 15`, `spaces = 20 - 15 = 5`, `gaps = 3`.
- `divmod(5, 3) = (1, 2)` → khe 0 và khe 1 nhận `2`, khe 2 nhận `1`.
- Kết quả: `a` + 2 cách + `computer.` + 2 cách + `Art` + 1 cách + `is` = `"a  computer.  Art is"`.

In [10]:
line = ["a", "computer.", "Art", "is"]
maxWidth = 20
letters = sum(len(w) for w in line)
spaces = maxWidth - letters
gaps = len(line) - 1
q, r = divmod(spaces, gaps)
print("letters =", letters, "| spaces =", spaces, "| gaps =", gaps, "| q =", q, "| r =", r)
print("số dấu cách từng khe:", [q + 1 if i < r else q for i in range(gaps)])

letters = 15 | spaces = 5 | gaps = 3 | q = 1 | r = 2
số dấu cách từng khe: [2, 2, 1]


### Code Bước 2: `justify_line`

Ta gom các mảnh (từ, khoảng trắng, từ, khoảng trắng, ...) vào một list `pieces` rồi `''.join(pieces)` một lần ở cuối. Đây là thói quen tốt trong Python: nối chuỗi bằng `join` thay vì `+=` lặp nhiều lần.

Hàm này **chỉ dành cho dòng có ít nhất 2 từ** (vì cần `gaps >= 1` để `divmod` không chia cho 0).

In [11]:
def justify_line(line: List[str], maxWidth: int) -> str:
    # Giả định: len(line) >= 2 và không phải dòng cuối
    letters = sum(len(w) for w in line)
    spaces = maxWidth - letters
    gaps = len(line) - 1
    q, r = divmod(spaces, gaps)

    pieces = []
    for i in range(gaps):
        pieces.append(line[i])
        if i < r:
            pieces.append(" " * (q + 1))   # khe bên trái nhận thêm 1
        else:
            pieces.append(" " * q)
    pieces.append(line[-1])                # từ cuối không có khe phía sau
    return "".join(pieces)


show([
    justify_line(["This", "is", "an"], 16),
    justify_line(["example", "of", "text"], 16),
    justify_line(["a", "computer.", "Art", "is"], 20),
])

|This    is    an|  len = 16
|example  of text|  len = 16
|a  computer.  Art is|  len = 20


### Trace Bước 2 trên `["a", "computer.", "Art", "is"]`, `maxWidth = 20`

`q = 1`, `r = 2`:

| Khe i | Từ bên trái khe | `i < r`? | Số dấu cách | Chuỗi tích lũy |
|---:|---|---|---:|---|
| 0 | `a` | `0 < 2` đúng | 2 | `\|a  \|` |
| 1 | `computer.` | `1 < 2` đúng | 2 | `\|a  computer.  \|` |
| 2 | `Art` | `2 < 2` sai | 1 | `\|a  computer.  Art \|` |
| — | `is` (từ cuối) | — | 0 | `\|a  computer.  Art is\|` |

In [12]:
def trace_justify_line(line: List[str], maxWidth: int) -> str:
    letters = sum(len(w) for w in line)
    spaces = maxWidth - letters
    gaps = len(line) - 1
    q, r = divmod(spaces, gaps)
    print(f"letters={letters}, spaces={spaces}, gaps={gaps}, divmod -> q={q}, r={r}")
    built = ""
    for i in range(gaps):
        k = q + 1 if i < r else q
        built += line[i] + " " * k
        print(f"khe {i}: từ {line[i]!r:12} i<r? {str(i < r):5} -> {k} dấu cách | |{built}|")
    built += line[-1]
    print(f"thêm từ cuối {line[-1]!r}            | |{built}|  len={len(built)}")
    return built


_ = trace_justify_line(["a", "computer.", "Art", "is"], 20)

letters=15, spaces=5, gaps=3, divmod -> q=1, r=2
khe 0: từ 'a'          i<r? True  -> 2 dấu cách | |a  |
khe 1: từ 'computer.'  i<r? True  -> 2 dấu cách | |a  computer.  |
khe 2: từ 'Art'        i<r? False -> 1 dấu cách | |a  computer.  Art |
thêm từ cuối 'is'            | |a  computer.  Art is|  len=20


## 6. Bước 3 — Hai trường hợp đặc biệt: căn trái

Có **hai loại dòng** không được căn đều mà phải **căn trái** (các từ cách nhau đúng 1 dấu cách, đệm phần còn lại ở cuối):

1. **Dòng cuối cùng** — đề nói rõ.
2. **Dòng chỉ có 1 từ** — không có khe nào để rải khoảng trắng, nên toàn bộ khoảng trắng nằm sau từ đó. (Ví dụ 2: `"acknowledgment  "`.)

Nếu quên trường hợp 2 và gọi thẳng `justify_line(["acknowledgment"], 16)`, ta có `gaps = 0` → `divmod(2, 0)` → **lỗi chia cho 0**:

In [13]:
try:
    justify_line(["acknowledgment"], 16)
except ZeroDivisionError as e:
    print("Lỗi:", type(e).__name__, "-", e)

Lỗi: ZeroDivisionError - integer division or modulo by zero


### Code Bước 3: `left_justify_line`

Rất ngắn: nối các từ bằng 1 dấu cách, rồi đệm dấu cách ở cuối cho đủ `maxWidth`. Hàm này dùng được cho cả dòng cuối lẫn dòng 1 từ (với 1 từ, `' '.join` trả lại đúng từ đó).

Đề đảm bảo dòng nào cũng có `len(' '.join(line)) <= maxWidth` (vì Bước 1 đã kiểm tra), nên phần đệm không bao giờ âm.

In [14]:
def left_justify_line(line: List[str], maxWidth: int) -> str:
    s = " ".join(line)
    return s + " " * (maxWidth - len(s))     # tương đương: s.ljust(maxWidth)


show([
    left_justify_line(["acknowledgment"], 16),   # dòng 1 từ
    left_justify_line(["shall", "be"], 16),      # dòng cuối
    left_justify_line(["do"], 20),
])

|acknowledgment  |  len = 16
|shall be        |  len = 16
|do                  |  len = 20


## 7. Cách 1 — Ghép 3 bước thành lời giải hoàn chỉnh (cách chính)

Giờ chỉ cần duyệt các dòng từ Bước 1 và chọn hàm phù hợp:

```text
nếu là dòng cuối HOẶC dòng chỉ có 1 từ  → left_justify_line
ngược lại                                → justify_line
```

`enumerate(lines)` cho ta cả chỉ số `idx` lẫn phần tử `line` trong mỗi vòng lặp; dòng cuối là dòng có `idx == len(lines) - 1`.

In [15]:
def full_justify_clear(words: List[str], maxWidth: int) -> List[str]:
    lines = split_into_lines(words, maxWidth)            # Bước 1
    result = []
    for idx, line in enumerate(lines):
        is_last = idx == len(lines) - 1
        if is_last or len(line) == 1:                    # Bước 3
            result.append(left_justify_line(line, maxWidth))
        else:                                            # Bước 2
            result.append(justify_line(line, maxWidth))
    return result


words_2 = ["What", "must", "be", "acknowledgment", "shall", "be"]
words_3 = ["Science", "is", "what", "we", "understand", "well", "enough", "to",
           "explain", "to", "a", "computer.", "Art", "is", "everything", "else", "we", "do"]

show(full_justify_clear(words_1, 16)); print()
show(full_justify_clear(words_2, 16)); print()
show(full_justify_clear(words_3, 20))

|This    is    an|  len = 16
|example  of text|  len = 16
|justification.  |  len = 16

|What   must   be|  len = 16
|acknowledgment  |  len = 16
|shall be        |  len = 16

|Science  is  what we|  len = 20
|understand      well|  len = 20
|enough to explain to|  len = 20
|a  computer.  Art is|  len = 20
|everything  else  we|  len = 20
|do                  |  len = 20


### Code nộp LeetCode — Cách 1 (tách hàm rõ ràng)

LeetCode cho phép viết thêm phương thức phụ trong `class Solution`. Ta giữ nguyên 3 bước, mỗi bước một phương thức.

Ngay sau đó là ô test gồm 3 ví dụ của đề và các edge case:

- Chỉ 1 từ, dài đúng `maxWidth` (không cần đệm).
- `maxWidth = 1` với các từ 1 ký tự (mỗi từ một dòng).
- Mọi từ vừa khít một dòng duy nhất (dòng đó cũng là dòng cuối → căn trái).
- Dòng giữa chỉ có 1 từ, xen giữa các dòng nhiều từ.

In [16]:
class Solution:
    def fullJustify(self, words: List[str], maxWidth: int) -> List[str]:
        lines = self._split_into_lines(words, maxWidth)
        result = []
        for idx, line in enumerate(lines):
            if idx == len(lines) - 1 or len(line) == 1:
                result.append(" ".join(line).ljust(maxWidth))
            else:
                result.append(self._justify_line(line, maxWidth))
        return result

    def _split_into_lines(self, words: List[str], maxWidth: int) -> List[List[str]]:
        lines, cur, letters = [], [], 0
        for w in words:
            if letters + len(w) + len(cur) > maxWidth:
                lines.append(cur)
                cur, letters = [], 0
            cur.append(w)
            letters += len(w)
        lines.append(cur)
        return lines

    def _justify_line(self, line: List[str], maxWidth: int) -> str:
        letters = sum(len(w) for w in line)
        gaps = len(line) - 1
        q, r = divmod(maxWidth - letters, gaps)
        pieces = []
        for i in range(gaps):
            pieces.append(line[i])
            pieces.append(" " * (q + 1 if i < r else q))
        pieces.append(line[-1])
        return "".join(pieces)

In [17]:
expected_2 = ["What   must   be", "acknowledgment  ", "shall be        "]
expected_3 = [
    "Science  is  what we",
    "understand      well",
    "enough to explain to",
    "a  computer.  Art is",
    "everything  else  we",
    "do                  ",
]

sol = Solution()
tests = [
    (words_1, 16, expected_1),
    (words_2, 16, expected_2),
    (words_3, 20, expected_3),
    (["hello"], 5, ["hello"]),
    (["a", "b", "c"], 1, ["a", "b", "c"]),
    (["a", "b", "c"], 5, ["a b c"]),
    (["ab", "cd", "longword", "e", "f"], 8, ["ab    cd", "longword", "e f     "]),
    (["Listen", "to", "many,", "speak", "to", "a", "few."], 6,
     ["Listen", "to    ", "many, ", "speak ", "to   a", "few.  "]),
]
for words, width, expected in tests:
    got = sol.fullJustify(words, width)
    assert got == expected, (words, width, got)
    assert got == full_justify_clear(words, width)
print("Tất cả", len(tests), "test của class Solution đều PASS!")

Tất cả 8 test của class Solution đều PASS!


### Độ phức tạp của Cách 1

Đặt `n = len(words)`, `W = maxWidth`, và `L` là số dòng kết quả (`L <= n`).

- **Bước 1**: mỗi từ được xét đúng 1 lần, mỗi lần O(1) → `O(n)`.
- **Bước 2 + 3**: mỗi dòng được dựng bằng `join` trên tổng cộng `W` ký tự → `O(W)` mỗi dòng → `O(L · W)`.
- **Thời gian**: `O(n + L · W)`, tức `O(n · W)` trong trường hợp xấu. Không thể tốt hơn đáng kể vì riêng việc **viết ra** kết quả đã tốn `L · W` ký tự.
- **Bộ nhớ**: `O(L · W)` cho kết quả (bắt buộc phải có) và `O(n)` cho list `lines` trung gian.

## 8. Cách 2 — Bản gọn round-robin

Cách 1 rõ ràng nhưng dài. Câu hỏi: có cách nào **không cần tính `q, r`** mà vẫn đảm bảo "khe trái nhận nhiều hơn"?

**Ý tưởng round-robin** (chia bài vòng tròn): có `spaces` dấu cách, ta phát **từng dấu cách một** cho các khe theo thứ tự khe 0, khe 1, ..., khe cuối, rồi quay lại khe 0, ... — giống như chia bài cho người chơi ngồi quanh bàn.

- Mỗi khe nhận được `spaces // gaps` vòng đầy đủ.
- Vòng cuối bị dở dang `spaces % gaps` lá — và những lá này rơi vào **các khe đầu tiên**, tức là bên trái!

Vậy round-robin cho **đúng cùng kết quả** với `divmod`, mà không cần tính toán gì.

Cài đặt: thay vì lưu khe riêng, ta **dán dấu cách vào đuôi từ** đứng trước khe: khe `i` nằm sau `line[i]`, nên `line[i % gaps] += ' '`. Với dòng 1 từ, `gaps = 0`, nên dùng mẹo `len(line) - 1 or 1` → chia vòng cho 1, mọi dấu cách dán vào sau từ duy nhất — đúng là căn trái!

In [18]:
# Thí nghiệm nhỏ: phát 5 dấu cách cho 3 khe theo vòng
gaps_count = [0, 0, 0]
for i in range(5):
    gaps_count[i % 3] += 1
    print(f"dấu cách thứ {i} -> khe {i % 3} | số dấu cách mỗi khe: {gaps_count}")
print("So với divmod(5, 3) =", divmod(5, 3), "-> [q+1, q+1, q] =", [2, 2, 1])

dấu cách thứ 0 -> khe 0 | số dấu cách mỗi khe: [1, 0, 0]
dấu cách thứ 1 -> khe 1 | số dấu cách mỗi khe: [1, 1, 0]
dấu cách thứ 2 -> khe 2 | số dấu cách mỗi khe: [1, 1, 1]
dấu cách thứ 3 -> khe 0 | số dấu cách mỗi khe: [2, 1, 1]
dấu cách thứ 4 -> khe 1 | số dấu cách mỗi khe: [2, 2, 1]
So với divmod(5, 3) = (1, 2) -> [q+1, q+1, q] = [2, 2, 1]


### Code Cách 2: `full_justify_round_robin`

Chú ý điều kiện chia dòng `width + len(w) + len(line) > maxWidth` giống hệt Bước 1 (`width` ở đây là tổng chữ cái). Khác biệt: ta **dựng dòng ngay khi đóng dòng**, không lưu list các dòng trung gian. Dòng cuối được xử lý riêng sau vòng lặp bằng `ljust`.

In [19]:
def full_justify_round_robin(words: List[str], maxWidth: int) -> List[str]:
    res, line, width = [], [], 0          # width = tổng chữ cái trong line
    for w in words:
        if width + len(w) + len(line) > maxWidth:
            # phát (maxWidth - width) dấu cách theo vòng vào đuôi các từ
            for i in range(maxWidth - width):
                line[i % (len(line) - 1 or 1)] += " "
            res.append("".join(line))
            line, width = [], 0
        line.append(w)
        width += len(w)
    return res + [" ".join(line).ljust(maxWidth)]   # dòng cuối: căn trái


show(full_justify_round_robin(words_3, 20))
assert full_justify_round_robin(words_1, 16) == expected_1
assert full_justify_round_robin(words_2, 16) == expected_2
assert full_justify_round_robin(words_3, 20) == expected_3
print("Round-robin PASS 3 ví dụ của đề")

|Science  is  what we|  len = 20
|understand      well|  len = 20
|enough to explain to|  len = 20
|a  computer.  Art is|  len = 20
|everything  else  we|  len = 20
|do                  |  len = 20
Round-robin PASS 3 ví dụ của đề


### Trace round-robin trên `["a", "computer.", "Art", "is"]`, `maxWidth = 20`

`width = 15`, cần phát `20 - 15 = 5` dấu cách, `len(line) - 1 or 1 = 3`:

| i | `i % 3` | Từ được dán thêm 1 dấu cách | line sau bước này |
|---:|---:|---|---|
| 0 | 0 | `a` | `['a ', 'computer.', 'Art', 'is']` |
| 1 | 1 | `computer.` | `['a ', 'computer. ', 'Art', 'is']` |
| 2 | 2 | `Art` | `['a ', 'computer. ', 'Art ', 'is']` |
| 3 | 0 | `a` | `['a  ', 'computer. ', 'Art ', 'is']` |
| 4 | 1 | `computer.` | `['a  ', 'computer.  ', 'Art ', 'is']` |

Từ cuối `is` không bao giờ được dán (chỉ số chỉ chạy `0..2`), nên dòng không có dấu cách thừa ở cuối. `''.join` → `"a  computer.  Art is"`.

In [20]:
def trace_round_robin_line(line: List[str], maxWidth: int) -> str:
    line = list(line)                      # copy để không sửa list gốc
    width = sum(len(w) for w in line)
    mod = len(line) - 1 or 1
    print(f"width={width}, cần phát {maxWidth - width} dấu cách, mod={mod}")
    for i in range(maxWidth - width):
        line[i % mod] += " "
        print(f"i={i}, i % {mod} = {i % mod} -> {line}")
    result = "".join(line)
    print(f"Kết quả: |{result}|")
    return result


_ = trace_round_robin_line(["a", "computer.", "Art", "is"], 20)
print()
_ = trace_round_robin_line(["acknowledgment"], 16)   # dòng 1 từ: mod = 0 or 1 = 1

width=15, cần phát 5 dấu cách, mod=3
i=0, i % 3 = 0 -> ['a ', 'computer.', 'Art', 'is']
i=1, i % 3 = 1 -> ['a ', 'computer. ', 'Art', 'is']
i=2, i % 3 = 2 -> ['a ', 'computer. ', 'Art ', 'is']
i=3, i % 3 = 0 -> ['a  ', 'computer. ', 'Art ', 'is']
i=4, i % 3 = 1 -> ['a  ', 'computer.  ', 'Art ', 'is']
Kết quả: |a  computer.  Art is|

width=14, cần phát 2 dấu cách, mod=1
i=0, i % 1 = 0 -> ['acknowledgment ']
i=1, i % 1 = 0 -> ['acknowledgment  ']
Kết quả: |acknowledgment  |


### Độ phức tạp của Cách 2

- Chia dòng vẫn `O(n)`.
- Mỗi dòng phát tối đa `W` dấu cách; nhưng mỗi lệnh `line[i] += ' '` **tạo chuỗi mới** (chuỗi Python bất biến — immutable), sao chép lại cả từ đó. Về lý thuyết, một dòng có thể tốn tới `O(W²)`, tổng `O(L · W²)`.
- Với `W <= 100` thì con số này rất nhỏ, LeetCode chấp nhận thoải mái. Ưu điểm là code **rất ngắn**, dễ viết khi phỏng vấn; nhược điểm là khó đọc hơn nếu chưa hiểu mẹo `or 1` và round-robin.
- Bộ nhớ: `O(L · W)` cho kết quả, `O(W)` cho dòng đang xây.

## 9. Cách 3 — Bản gọn dùng chỉ số `i, j` + `divmod` + `ljust` + `join`

Cách 1 tạo hẳn list `lines` trung gian. Cách 3 làm tất cả trong **một vòng lặp**, dùng hai chỉ số:

- `i`: vị trí từ đầu tiên của dòng đang xét.
- `j`: tiến dần sang phải chừng nào còn nhét được; dừng thì dòng gồm `words[i:j]`.

Điều kiện nhét được viết theo chỉ số: dòng hiện có `j - i` từ, nên thêm `words[j]` cần tối thiểu `letters + len(words[j]) + (j - i) <= maxWidth` — y hệt Bước 1.

Dựng dòng bình thường bằng một **list khoảng trắng** `gaps`:

```python
gaps = [' ' * (q + 1)] * r + [' ' * q] * (len(line) - 1 - r)
```

(`[x] * k` là list gồm `k` phần tử `x` — phép `*` cũng lặp được list!) rồi ghép từ với khe bằng `zip`.

Hai công cụ mới cần thử trước: **slicing** `words[i:j]` và **`zip`**.

In [21]:
demo = ["a", "computer.", "Art", "is", "everything"]
print(demo[0:4])          # các phần tử chỉ số 0, 1, 2, 3 (không gồm 4)

print([" " * 2] * 2 + [" " * 1] * 1)      # list khoảng trắng: ['  ', '  ', ' ']

# zip ghép từng cặp phần tử cùng vị trí của hai list
for w, g in zip(["a", "computer.", "Art", "is"], ["  ", "  ", " ", ""]):
    print(f"|{w}|{g}|")

['a', 'computer.', 'Art', 'is']
['  ', '  ', ' ']
|a|  |
|computer.|  |
|Art| |
|is||


In [22]:
def full_justify_compact(words: List[str], maxWidth: int) -> List[str]:
    res = []
    i, n = 0, len(words)
    while i < n:
        # mở rộng j chừng nào còn nhét được words[j]
        j, letters = i, 0
        while j < n and letters + len(words[j]) + (j - i) <= maxWidth:
            letters += len(words[j])
            j += 1
        line = words[i:j]

        if j == n or len(line) == 1:                 # dòng cuối hoặc 1 từ
            res.append(" ".join(line).ljust(maxWidth))
        else:
            q, r = divmod(maxWidth - letters, len(line) - 1)
            gaps = [" " * (q + 1)] * r + [" " * q] * (len(line) - 1 - r)
            res.append("".join(w + g for w, g in zip(line, gaps + [""])))
        i = j                                        # dòng sau bắt đầu từ j
    return res


show(full_justify_compact(words_2, 16))
assert full_justify_compact(words_1, 16) == expected_1
assert full_justify_compact(words_2, 16) == expected_2
assert full_justify_compact(words_3, 20) == expected_3
print("Compact PASS 3 ví dụ của đề")

|What   must   be|  len = 16
|acknowledgment  |  len = 16
|shall be        |  len = 16
Compact PASS 3 ví dụ của đề


Ghi chú về Cách 3:

- `j == n` nghĩa là dòng này "ăn" tới hết mảng → đó là dòng cuối. Ta không cần biết trước tổng số dòng như Cách 1.
- `gaps + [""]`: thêm một khe rỗng cho từ cuối, để `zip` ghép đủ mọi từ (nếu không, `zip` dừng ở list ngắn hơn và **mất từ cuối**).
- Vòng `while` trong luôn tăng `j` ít nhất 1 lần (vì mỗi từ dài `<= maxWidth`), nên vòng ngoài không bị lặp vô hạn.
- Độ phức tạp giống Cách 1: thời gian `O(n + L · W)`, bộ nhớ `O(L · W)` cho kết quả và `O(W)` phụ.

## 10. Kiểm tra chéo bằng "trọng tài" theo định nghĩa

Bài mô phỏng không có brute force để so sánh. Thay vào đó, ta viết một hàm **trọng tài** `check_justified` kiểm tra kết quả theo **đúng từng câu của đề**, hoàn toàn độc lập với cách dựng:

1. Mỗi dòng dài đúng `maxWidth`, không bắt đầu bằng dấu cách.
2. Đọc lại các từ trong các dòng (bằng `line.split()`) phải ra đúng `words` theo thứ tự.
3. **Greedy**: với mọi dòng (trừ dòng cuối), từ đầu của dòng sau **không thể** nhét thêm vào dòng này.
4. Dòng cuối và dòng 1 từ: các từ cách nhau đúng 1 dấu cách, phần còn lại là dấu cách ở cuối.
5. Dòng bình thường: không có dấu cách thừa ở cuối; các khe chênh nhau tối đa 1; khe trái `>=` khe phải.

`line.split()` (không truyền tham số) tách theo **mọi** đoạn khoảng trắng, nên `"a  computer.  Art is".split()` → `['a', 'computer.', 'Art', 'is']`.

Sau đó ta sinh ngẫu nhiên nhiều input nhỏ (cố định `random.seed` để lần chạy nào cũng giống nhau) và kiểm tra: cả 3 cách cho **cùng một kết quả**, trùng với `class Solution`, và kết quả được trọng tài chấp nhận.

In [23]:
def gap_sizes(line: str) -> List[int]:
    # Độ dài các đoạn dấu cách nằm GIỮA hai từ (bỏ đoạn cuối dòng)
    sizes, count = [], 0
    for ch in line.rstrip(" "):
        if ch == " ":
            count += 1
        elif count > 0:
            sizes.append(count)
            count = 0
    return sizes


def check_justified(words: List[str], maxWidth: int, lines: List[str]) -> bool:
    groups = [line.split() for line in lines]
    if [w for g in groups for w in g] != words:                 # (2) đúng từ, đúng thứ tự
        return False
    for idx, (line, g) in enumerate(zip(lines, groups)):
        if len(line) != maxWidth or line[0] == " ":             # (1)
            return False
        is_last = idx == len(lines) - 1
        if not is_last and len(" ".join(g + [groups[idx + 1][0]])) <= maxWidth:
            return False                                        # (3) chưa đủ tham lam
        if is_last or len(g) == 1:                              # (4)
            if line != " ".join(g).ljust(maxWidth):
                return False
        else:                                                   # (5)
            sizes = gap_sizes(line)
            if line[-1] == " " or max(sizes) - min(sizes) > 1 or sizes != sorted(sizes, reverse=True):
                return False
    return True


print(gap_sizes("a  computer.  Art is"))                       # [2, 2, 1]
print(check_justified(words_3, 20, expected_3))                # True
print(check_justified(words_1, 16, ["This    is    an", "example of  text", "justification.  "]))  # False: dư dồn sai bên

[2, 2, 1]
True
False


In [24]:
import random

random.seed(68)
letters_pool = "abcdefghij.,!"
num_tests = 500
for _ in range(num_tests):
    maxWidth = random.randint(1, 20)
    n = random.randint(1, 12)
    words = ["".join(random.choice(letters_pool) for _ in range(random.randint(1, min(maxWidth, 6))))
             for _ in range(n)]
    a = full_justify_clear(words, maxWidth)
    b = full_justify_round_robin(words, maxWidth)
    c = full_justify_compact(words, maxWidth)
    d = Solution().fullJustify(words, maxWidth)
    assert a == b == c == d, (words, maxWidth, a, b, c, d)
    assert check_justified(words, maxWidth, a), (words, maxWidth, a)
print(f"{num_tests} test ngẫu nhiên: 3 cách + Solution trùng nhau và đều hợp lệ theo trọng tài!")

500 test ngẫu nhiên: 3 cách + Solution trùng nhau và đều hợp lệ theo trọng tài!


## 11. Vì sao thuật toán đúng?

**Bước 1 (chia dòng)** — bất biến (invariant) của vòng lặp:

> Tại mọi thời điểm, `cur` là **tiền tố dài nhất** của phần từ còn lại mà xếp được vào một dòng, nếu mỗi khe chỉ dùng 1 dấu cách: `letters + (len(cur) - 1) <= maxWidth`.

Ta chỉ thêm `w` khi điều kiện `letters + len(w) + len(cur) <= maxWidth` đúng, nên bất biến được giữ. Khi điều kiện sai, `w` thật sự không vừa (kể cả với số dấu cách tối thiểu), nên đóng dòng là đúng tinh thần greedy của đề.

**Bước 2 (phân phối)**:

- Tổng dấu cách: `r·(q+1) + (gaps−r)·q = gaps·q + r = spaces` → dòng dài đúng `maxWidth`.
- Các khe chỉ nhận `q` hoặc `q+1` → chênh lệch tối đa 1, tức là **đều nhất có thể**.
- Các khe `q+1` nằm ở đầu (chỉ số `< r`) → **khe trái nhiều hơn khe phải**.

**Round-robin = divmod**: sau khi phát `spaces` dấu cách theo vòng cho `gaps` khe, khe `i` nhận `spaces // gaps` lá từ các vòng đầy đủ, cộng thêm 1 nếu `i < spaces % gaps` (vòng dở dang đi từ khe 0). Đó chính xác là `q + (1 if i < r else 0)`.

**Bước 3**: dòng cuối và dòng 1 từ được căn trái đúng nghĩa đen của đề; `gaps = 0` không bao giờ đi vào `divmod`.

## 12. So sánh 3 cách

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Khi nào nên dùng |
|---|---|---|---|---|
| 1. Tách hàm | `split_into_lines` → `justify_line` (divmod) / `left_justify_line` | `O(n + L·W)` | `O(n)` (list dòng) | Học hiểu, phỏng vấn cần giải thích rõ, dễ debug |
| 2. Round-robin | Phát từng dấu cách `line[i % (len(line)-1 or 1)] += ' '` | `O(n + L·W²)` lý thuyết (copy chuỗi) | `O(W)` | Viết nhanh, code ngắn nhất |
| 3. Chỉ số + `ljust`/`join` | Hai chỉ số `i, j`, `divmod`, list `gaps`, `zip` | `O(n + L·W)` | `O(W)` | Gọn mà vẫn tường minh, không cần list dòng trung gian |

Trong đó `n = len(words)`, `W = maxWidth`, `L` = số dòng kết quả. Cả ba đều thừa nhanh với constraints (`n <= 300`, `W <= 100`).

Lời khuyên: **học Cách 1 trước** để nắm logic, sau đó ghi nhớ Cách 2 làm "template" nộp nhanh.

## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên đếm dấu cách tối thiểu khi chia dòng

Sai:

```python
if letters + len(w) > maxWidth:     # quên + len(cur)
```

Khi đó dòng có thể chứa quá nhiều từ: tổng chữ cái vừa đủ nhưng **không còn chỗ cho dấu cách**. Ở Ví dụ 1, dòng đầu sẽ thành `["This", "is", "an", "example"]` (15 chữ cái, cần 3 khe nhưng chỉ còn 1 dấu cách) → các từ bị **dính vào nhau**.

Sửa: `if letters + len(w) + len(cur) > maxWidth:`

In [25]:
def split_into_lines_wrong(words, maxWidth):
    lines, cur, letters = [], [], 0
    for w in words:
        if letters + len(w) > maxWidth:        # SAI: quên len(cur)
            lines.append(cur)
            cur, letters = [], 0
        cur.append(w)
        letters += len(w)
    lines.append(cur)
    return lines


bad_lines = split_into_lines_wrong(words_1, 16)
print("Dòng đầu (sai):", bad_lines[0])
print("Dựng thử:", "|" + justify_line(bad_lines[0], 16) + "|", "-> từ bị dính nhau!")

Dòng đầu (sai): ['This', 'is', 'an', 'example']
Dựng thử: |This isanexample| -> từ bị dính nhau!


### Lỗi 2: Dồn khoảng thừa sang bên phải

Sai: cho `r` khe **cuối** nhận thêm (`i >= gaps - r`). Đề yêu cầu bên **trái** nhận nhiều hơn.

### Lỗi 3: Căn đều cả dòng cuối

Sai: đưa dòng cuối vào `justify_line`. Ví dụ 2 sẽ ra `"shall         be"` thay vì `"shall be        "`.

### Lỗi 4: Căn trái dòng cuối nhưng quên đệm đủ `maxWidth`

Sai: `' '.join(line)` mà không `ljust(maxWidth)`. Dòng cuối ngắn hơn `maxWidth` → sai đề ("mỗi dòng có đúng `maxWidth` ký tự").

### Lỗi 5: Chia cho 0 ở dòng chỉ có 1 từ

Đã thấy ở Bước 3: `divmod(spaces, 0)` → `ZeroDivisionError`. Sửa: kiểm tra `len(line) == 1` trước (Cách 1, 3) hoặc dùng `len(line) - 1 or 1` (Cách 2).

### Lỗi 6: Quên dòng cuối sau vòng lặp

Vòng `for w in words` chỉ đóng dòng khi gặp từ **không vừa**. Dòng cuối không bao giờ gặp từ như vậy, nên nó vẫn nằm trong `cur`/`line` khi vòng lặp kết thúc. Phải xử lý nó **sau vòng lặp** (`lines.append(cur)` hoặc `res + [' '.join(line).ljust(maxWidth)]`).

### Lỗi 7: Trong round-robin, chia vòng cho `len(line)` thay vì `len(line) - 1`

Nếu dùng `i % len(line)`, dấu cách sẽ bị dán cả vào sau **từ cuối** → dòng có dấu cách thừa ở cuối, các khe giữa bị thiếu.

Ô dưới đây minh họa Lỗi 2, 3, 4 bằng ký hiệu `|`:

In [26]:
def justify_line_right_heavy(line, maxWidth):          # Lỗi 2
    letters = sum(len(w) for w in line)
    gaps = len(line) - 1
    q, r = divmod(maxWidth - letters, gaps)
    pieces = []
    for i in range(gaps):
        pieces.append(line[i])
        pieces.append(" " * (q + 1 if i >= gaps - r else q))   # SAI: dồn phải
    pieces.append(line[-1])
    return "".join(pieces)


print("Lỗi 2 :", "|" + justify_line_right_heavy(["example", "of", "text"], 16) + "|")
print("Đúng  :", "|" + justify_line(["example", "of", "text"], 16) + "|")
print("Lỗi 3 :", "|" + justify_line(["shall", "be"], 16) + "|")
print("Lỗi 4 :", "|" + " ".join(["shall", "be"]) + "|", "len =", len("shall be"))
print("Đúng  :", "|" + " ".join(["shall", "be"]).ljust(16) + "|")

Lỗi 2 : |example of  text|
Đúng  : |example  of text|
Lỗi 3 : |shall         be|
Lỗi 4 : |shall be| len = 8
Đúng  : |shall be        |


## 14. Mẹo nhận dạng pattern và bài liên quan

**Nhận dạng bài mô phỏng**:

- Đề mô tả một **quy trình** rất cụ thể (như phần mềm soạn thảo căn lề), nhiều quy tắc "nếu… thì…", không hỏi "tối ưu" hay "đếm số cách".
- Constraints nhỏ (`n <= 300`, `W <= 100`) → không cần thuật toán phức tạp, cái khó là **không sai chi tiết**.

**Chiến lược chung cho bài mô phỏng**:

1. Viết ra tay một ví dụ, ghi lại các đại lượng (ở đây: chữ cái, số khe, `q`, `r`).
2. Tách thành các bước nhỏ, mỗi bước một hàm, test từng hàm.
3. Liệt kê trường hợp đặc biệt (dòng cuối, 1 từ, từ dài đúng `maxWidth`).
4. Viết trọng tài kiểm tra theo định nghĩa + test ngẫu nhiên.

**Công thức "chia đều có dư"** `q, r = divmod(total, k)` rồi cho `r` phần đầu nhận `q + 1` xuất hiện ở rất nhiều bài: chia mảng thành `k` phần gần bằng nhau, phân trang, chia việc cho máy…

**Bài liên quan / biến thể**:

| Bài | Liên hệ |
|---|---|
| LeetCode 1592 — Rearrange Spaces Between Words | Chia đều dấu cách giữa các từ bằng `divmod`, phần dư đặt cuối câu |
| LeetCode 418 — Sentence Screen Fitting | Cũng xếp từ greedy vào các dòng độ rộng cố định, đếm số lần câu vừa màn hình |
| LeetCode 2468 — Split Message Based on Limit | Chia chuỗi thành các phần có độ dài giới hạn, mô phỏng tỉ mỉ |
| Biến thể tự chế | Căn giữa mỗi dòng; căn phải; dồn khoảng thừa về bên phải |

## 15. Bài tập vận dụng

Tự làm trước, gợi ý ở mục 16, đáp án tham khảo ở mục 17 (cuối notebook).

### Bài 1 — Đoán output (không chạy code)

Đoán chính xác từng dòng in ra:

```python
print('|' + ' '.join(['to', 'be']) + '|')
print('|' + '--'.join(['a']) + '|')
print(divmod(5, 3))
print('|' + 'ab'.ljust(5) + '|')
print('|' + 'abcdef'.ljust(3) + '|')
print(0 or 1, 2 or 1)
print([i % 3 for i in range(5)])
```

### Bài 2 — Tính bằng tay

Với `words = ["Listen", "to", "many,", "speak", "to", "a", "few."]`, `maxWidth = 6`:

1. Lập bảng Bước 1: dòng nào gồm những từ nào?
2. Dòng nào là dòng bình thường (≥ 2 từ, không phải dòng cuối)? Tính `spaces`, `gaps`, `q`, `r` cho dòng đó.
3. Viết ra kết quả, kẹp mỗi dòng bằng `|`.

### Bài 3 — Điền TODO: phân phối khoảng trắng

Hoàn thiện `justify_line_exercise(line, maxWidth)` cho dòng có ít nhất 2 từ (không phải dòng cuối).

In [27]:
def justify_line_exercise(line: List[str], maxWidth: int) -> str:
    # TODO 1: tính letters = tổng chữ cái
    # TODO 2: spaces = maxWidth - letters, gaps = len(line) - 1
    # TODO 3: q, r = divmod(spaces, gaps)
    # TODO 4: duyệt các khe i, khe i < r nhận q + 1 dấu cách, còn lại nhận q
    # TODO 5: ghép từ cuối, trả về chuỗi
    pass

# Gợi ý test sau khi làm xong:
# assert justify_line_exercise(["This", "is", "an"], 16) == "This    is    an"
# assert justify_line_exercise(["example", "of", "text"], 16) == "example  of text"
# assert justify_line_exercise(["a", "computer.", "Art", "is"], 20) == "a  computer.  Art is"

### Bài 4 — Tự viết lại toàn bộ lời giải

Không nhìn code phía trên, tự viết `full_justify_exercise(words, maxWidth)` (cách nào cũng được). Nhớ cả 3 bước và 2 trường hợp đặc biệt.

In [28]:
def full_justify_exercise(words: List[str], maxWidth: int) -> List[str]:
    # TODO: Bước 1 chia dòng greedy (nhớ + số khe tối thiểu)
    # TODO: Bước 2 dòng bình thường: divmod, khe trái nhận thêm
    # TODO: Bước 3 dòng cuối / dòng 1 từ: căn trái + đệm đủ maxWidth
    raise NotImplementedError

# Gợi ý test:
# assert full_justify_exercise(words_1, 16) == expected_1
# assert full_justify_exercise(words_2, 16) == expected_2
# assert full_justify_exercise(words_3, 20) == expected_3
# assert full_justify_exercise(["hello"], 5) == ["hello"]

### Bài 5 — Biến thể: LeetCode 1592 Rearrange Spaces Between Words

Cho chuỗi `text` gồm các từ và dấu cách (có ít nhất 1 từ). Hãy sắp xếp lại các dấu cách sao cho số dấu cách giữa **mọi cặp từ kề nhau bằng nhau và lớn nhất có thể**. Nếu không chia hết, các dấu cách dư đặt ở **cuối chuỗi**. Chuỗi kết quả có cùng độ dài với `text`.

Ví dụ:

```text
"  this   is  a sentence "  ->  "this   is   a   sentence"
" practice   makes   perfect" -> "practice   makes   perfect "
"hello   "                  ->  "hello   "
```

In [29]:
def reorder_spaces(text: str) -> str:
    # TODO 1: đếm tổng số dấu cách trong text (text.count(' '))
    # TODO 2: lấy danh sách từ bằng text.split()
    # TODO 3: nếu chỉ 1 từ -> mọi dấu cách đặt ở cuối
    # TODO 4: q, r = divmod(số dấu cách, số khe); nối bằng ' ' * q, thêm ' ' * r ở cuối
    pass

# Gợi ý test:
# assert reorder_spaces("  this   is  a sentence ") == "this   is   a   sentence"
# assert reorder_spaces(" practice   makes   perfect") == "practice   makes   perfect "
# assert reorder_spaces("hello   ") == "hello   "

### Bài 6 — Mini challenge: căn giữa

Viết `center_justify(words, maxWidth)`:

- Chia dòng greedy **giống hệt** bài gốc.
- Mỗi dòng: nối các từ bằng **đúng 1 dấu cách**, rồi căn giữa: `extra = maxWidth - len(s)`, đệm `extra // 2` dấu cách bên trái và phần còn lại bên phải.
- Áp dụng cho mọi dòng, kể cả dòng cuối.

Ví dụ: `center_justify(["This", "is", "an", "example"], 10)` → `["This is an", " example  "]`.

In [30]:
def center_justify(words: List[str], maxWidth: int) -> List[str]:
    # TODO 1: tái sử dụng logic chia dòng greedy
    # TODO 2: với mỗi dòng: s = ' '.join(line), extra = maxWidth - len(s)
    # TODO 3: left = extra // 2, right = extra - left
    pass

# Gợi ý test:
# assert center_justify(["This", "is", "an", "example"], 10) == ["This is an", " example  "]
# assert center_justify(["a", "b"], 4) == ["a b "]

## 16. Gợi ý theo từng mức

**Bài 1**
- Mức 1: nhớ lại `join` chèn chuỗi ngăn cách vào **giữa**; list 1 phần tử thì không có "giữa".
- Mức 2: `ljust` không bao giờ cắt chuỗi; `x or y` trả `x` nếu `x` khác 0.
- Mức 3: `divmod(5, 3)` là `(5 // 3, 5 % 3)`.

**Bài 2**
- Mức 1: với `maxWidth = 6`, hai từ `"to"` + `"many,"` cần tối thiểu `2 + 5 + 1 = 8` ký tự.
- Mức 2: chỉ có một dòng bình thường; tất cả các dòng khác chỉ có 1 từ hoặc là dòng cuối.
- Mức 3: dòng `["to", "a"]`: `letters = 3`, `spaces = 3`, `gaps = 1`.

**Bài 3**
- Mức 1: số khe luôn là `len(line) - 1`.
- Mức 2: `' ' * (q + 1 if i < r else q)`.
- Mức 3: gom vào `pieces`, nhớ `pieces.append(line[-1])` rồi `''.join(pieces)`.

**Bài 4**
- Mức 1: viết 3 hàm nhỏ như Cách 1 rồi ghép.
- Mức 2: điều kiện đóng dòng `letters + len(w) + len(cur) > maxWidth`.
- Mức 3: sau vòng lặp nhớ xử lý `cur` (dòng cuối) bằng `' '.join(cur).ljust(maxWidth)`.

**Bài 5**
- Mức 1: đây là `divmod` y như Bước 2, chỉ khác chỗ đặt phần dư.
- Mức 2: `len(words) - 1` có thể bằng 0 → xử lý riêng trường hợp 1 từ.
- Mức 3: `(' ' * q).join(words) + ' ' * r`.

**Bài 6**
- Mức 1: chỉ thay Bước 2 + Bước 3 bằng một công thức căn giữa.
- Mức 2: `' ' * left + s + ' ' * right`.
- Mức 3: dùng lại `split_into_lines` để có các dòng.

## 17. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1 — chạy để kiểm tra dự đoán

```text
|to be|
|a|
(1, 2)
|ab   |
|abcdef|
1 2
[0, 1, 2, 0, 1]
```

In [31]:
print('|' + ' '.join(['to', 'be']) + '|')
print('|' + '--'.join(['a']) + '|')
print(divmod(5, 3))
print('|' + 'ab'.ljust(5) + '|')
print('|' + 'abcdef'.ljust(3) + '|')
print(0 or 1, 2 or 1)
print([i % 3 for i in range(5)])

|to be|
|a|
(1, 2)
|ab   |
|abcdef|
1 2
[0, 1, 2, 0, 1]


### Đáp án Bài 2 — tính tay

| Dòng | Các từ | Loại | Tính toán | Kết quả |
|---|---|---|---|---|
| 1 | `Listen` | 1 từ | `"to"` thêm vào cần `6 + 2 + 1 = 9 > 6` | `\|Listen\|` |
| 2 | `to` | 1 từ | `"many,"` cần `2 + 5 + 1 = 8 > 6` | `\|to    \|` |
| 3 | `many,` | 1 từ | `"speak"` cần `5 + 5 + 1 = 11 > 6` | `\|many, \|` |
| 4 | `speak` | 1 từ | `"to"` cần `5 + 2 + 1 = 8 > 6` | `\|speak \|` |
| 5 | `to a` | bình thường | `letters = 3`, `spaces = 3`, `gaps = 1`, `q = 3`, `r = 0` | `\|to   a\|` |
| 6 | `few.` | dòng cuối | căn trái, đệm 2 | `\|few.  \|` |

(`"few."` không vừa dòng 5: `3 + 4 + 2 = 9 > 6`.)

In [32]:
words_ex2 = ["Listen", "to", "many,", "speak", "to", "a", "few."]
result_ex2 = full_justify_clear(words_ex2, 6)
show(result_ex2)
assert result_ex2 == ["Listen", "to    ", "many, ", "speak ", "to   a", "few.  "]

|Listen|  len = 6
|to    |  len = 6
|many, |  len = 6
|speak |  len = 6
|to   a|  len = 6
|few.  |  len = 6


In [33]:
# ===== Đáp án Bài 3 – 6 =====
def answer_justify_line_exercise(line: List[str], maxWidth: int) -> str:
    letters = sum(len(w) for w in line)
    spaces = maxWidth - letters
    gaps = len(line) - 1
    q, r = divmod(spaces, gaps)
    pieces = []
    for i in range(gaps):
        pieces.append(line[i])
        pieces.append(" " * (q + 1 if i < r else q))
    pieces.append(line[-1])
    return "".join(pieces)


def answer_full_justify_exercise(words: List[str], maxWidth: int) -> List[str]:
    result, cur, letters = [], [], 0
    for w in words:
        if letters + len(w) + len(cur) > maxWidth:
            if len(cur) == 1:
                result.append(cur[0].ljust(maxWidth))
            else:
                result.append(answer_justify_line_exercise(cur, maxWidth))
            cur, letters = [], 0
        cur.append(w)
        letters += len(w)
    result.append(" ".join(cur).ljust(maxWidth))   # dòng cuối
    return result


def answer_reorder_spaces(text: str) -> str:
    total_spaces = text.count(" ")
    words = text.split()
    if len(words) == 1:
        return words[0] + " " * total_spaces
    q, r = divmod(total_spaces, len(words) - 1)
    return (" " * q).join(words) + " " * r


def answer_center_justify(words: List[str], maxWidth: int) -> List[str]:
    result = []
    for line in split_into_lines(words, maxWidth):
        s = " ".join(line)
        extra = maxWidth - len(s)
        left = extra // 2
        right = extra - left
        result.append(" " * left + s + " " * right)
    return result


assert answer_justify_line_exercise(["This", "is", "an"], 16) == "This    is    an"
assert answer_justify_line_exercise(["example", "of", "text"], 16) == "example  of text"
assert answer_justify_line_exercise(["a", "computer.", "Art", "is"], 20) == "a  computer.  Art is"

assert answer_full_justify_exercise(words_1, 16) == expected_1
assert answer_full_justify_exercise(words_2, 16) == expected_2
assert answer_full_justify_exercise(words_3, 20) == expected_3
assert answer_full_justify_exercise(["hello"], 5) == ["hello"]

assert answer_reorder_spaces("  this   is  a sentence ") == "this   is   a   sentence"
assert answer_reorder_spaces(" practice   makes   perfect") == "practice   makes   perfect "
assert answer_reorder_spaces("hello   ") == "hello   "

assert answer_center_justify(["This", "is", "an", "example"], 10) == ["This is an", " example  "]
assert answer_center_justify(["a", "b"], 4) == ["a b "]
show(answer_center_justify(["This", "is", "an", "example"], 10))

print("Đáp án tham khảo chạy PASS toàn bộ test!")

|This is an|  len = 10
| example  |  len = 10
Đáp án tham khảo chạy PASS toàn bộ test!


## 18. Template cuối cùng nên nhớ

**Bản tách hàm (dễ giải thích):**

```python
class Solution:
    def fullJustify(self, words: List[str], maxWidth: int) -> List[str]:
        # Bước 1: chia dòng greedy — nhớ + len(cur) (số dấu cách tối thiểu)
        lines, cur, letters = [], [], 0
        for w in words:
            if letters + len(w) + len(cur) > maxWidth:
                lines.append(cur)
                cur, letters = [], 0
            cur.append(w)
            letters += len(w)
        lines.append(cur)

        result = []
        for idx, line in enumerate(lines):
            # Bước 3: dòng cuối hoặc 1 từ -> căn trái
            if idx == len(lines) - 1 or len(line) == 1:
                result.append(" ".join(line).ljust(maxWidth))
                continue
            # Bước 2: divmod, khe trái nhận thêm
            gaps = len(line) - 1
            q, r = divmod(maxWidth - sum(len(w) for w in line), gaps)
            pieces = []
            for i in range(gaps):
                pieces.append(line[i] + " " * (q + 1 if i < r else q))
            pieces.append(line[-1])
            result.append("".join(pieces))
        return result
```

**Bản round-robin (ngắn nhất):**

```python
class Solution:
    def fullJustify(self, words: List[str], maxWidth: int) -> List[str]:
        res, line, width = [], [], 0
        for w in words:
            if width + len(w) + len(line) > maxWidth:
                for i in range(maxWidth - width):
                    line[i % (len(line) - 1 or 1)] += " "
                res.append("".join(line))
                line, width = [], 0
            line.append(w)
            width += len(w)
        return res + [" ".join(line).ljust(maxWidth)]
```

## 19. Checklist tự kiểm tra trước khi submit

- Điều kiện chia dòng có **cộng số dấu cách tối thiểu** (`+ len(cur)`) chưa?
- Hết vòng lặp, mình có **xử lý dòng cuối** đang nằm trong `cur`/`line` chưa?
- Dòng cuối có được **căn trái** (1 dấu cách giữa các từ) và **đệm đủ `maxWidth`** chưa?
- Dòng **chỉ có 1 từ** có tránh được chia cho 0 và được đệm ở cuối chưa?
- Khoảng thừa `r` có được dồn về **bên trái** (`i < r`) chưa?
- Dòng bình thường có bị **dư dấu cách ở cuối** không (nhất là bản round-robin: phải `% (len(line) - 1)`)?
- Đã test 3 ví dụ của đề, ca 1 từ dài đúng `maxWidth`, ca `maxWidth = 1` chưa?
- In kết quả kẹp giữa `|` để **nhìn thấy** dấu cách khi debug.